# Figure 7 ver8 검산 (출력 CSV ↔ Results_and_Legends_ver8.md ↔ 그림)
2026-11-17. `data/*.csv`(파이프라인 출력)만으로 원고 문장·범례의 수치를 다시 계산해 대조합니다.
원본 입력(`sample_level_*.csv`)부터의 재실행은 별도 단계입니다.
`DATA`를 ver8 폴더의 `data/` 경로로 바꾸면 그대로 실행됩니다.

In [ ]:
import os, numpy as np, pandas as pd
from scipy import stats
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
DATA = os.environ.get('V8_DATA', 'data')
rd = lambda f, **k: pd.read_csv(os.path.join(DATA, f), **k)
SEL, SC, CI = rd('selection_funnel_43fine.csv'), rd('credibility_scorecard_v8.csv'), rd('cohort_effect_sizes_ci_v8.csv')
HS, PW, MOD = rd('hspc_death_analyses_v8.csv'), rd('power_curves_v8.csv'), rd('modality_scale_shared27_and_raw_pct_v8.csv')
SCAN, SA, CLL = rd('cll_scan_43fine_v8.csv'), rd('sensitivity_14variants_all_v8.csv'), rd('cll_molecular_screen_bulk_donors.csv')
NS, RAW = rd('patient_normal_scores_noCLL_v8.csv', index_col=0), rd('patient_raw_pct_noCLL_v8.csv', index_col=0)
SH = rd('shared27_paired_pct_v8.csv')
issues = []
def check(label, ok, detail=''):
    print(('PASS ' if ok else 'FAIL ') + label + ('' if ok else f'  -> {detail}'))
    if not ok: issues.append((label, detail))

## 1. 코호트 크기와 Fig. 7a CLL 스크린

In [ ]:
R = NS[NS.src=='Real']; N = NS[NS.src=='New']
check('Real n=37, 7 deaths', (len(R), R.fatal.sum()) == (37, 7))
check('PBISC new n=29, 6 deaths', (len(N), N.fatal.sum()) == (29, 6))
g = CLL.groupby(['cohort_group','covid_positive_scope']).size(); print(g)
check('bulk donors 105 = 37 neg + 37 Real/Shared + 29 PBISC-only + 2 CLL',
      len(CLL)==105 and g[('Real / Shared donor',True)]==37 and g[('PBISC-only donor',True)]==29 and (g[('Real / Shared donor',False)]+g[('PBISC-only donor',False)])==37)
x = CLL[CLL.cohort_group.str.startswith('Excl')]; o = CLL[~CLL.cohort_group.str.startswith('Excl')]
check('CLL ROR1>4.2 & FMOD>3.4', (x.ROR1>4.2).all() and (x.FMOD>3.4).all())
check('no other donor above dashed lines (ROR1 2.5, FMOD 1.0)', (o.ROR1<2.5).all() and (o.FMOD<1.0).all(), f'max {o.ROR1.max():.2f}/{o.FMOD.max():.2f}')
check('shared donors = 27 (subset of Real 37)', len(SH)==27 and set(SH.patient_id) <= set(R.index))

## 2. 선별 퍼널 43 → 16 → 15 → 12 → 8

In [ ]:
cnt = [len(SEL)] + [int((SEL[k+'_sev']|SEL[k+'_death']).sum()) for k in ['S2','S3','S4','S5']]
check('funnel 43/16/15/12/8', cnt == [43,16,15,12,8], cnt)
s5 = sorted(SEL.feature[SEL.S5_sev|SEL.S5_death]); print('S5:', s5)
check('S3 removes NKT only', sorted(set(SEL.feature[SEL.S2_sev|SEL.S2_death]) - set(SEL.feature[SEL.S3_sev|SEL.S3_death])) == ['NKT'])
check('S4 removes CD14mono_dex/IFN, MAIT', sorted(set(SEL.feature[SEL.S3_sev|SEL.S3_death]) - set(SEL.feature[SEL.S4_sev|SEL.S4_death])) == ['CD14mono_IFN','CD14mono_dex','MAIT'])
S = SEL.set_index('feature')
for f, rho, q in [('NK CD56 bright',-0.50,3.7e-4),('B_ASC_dividing',0.48,6.1e-4),('CD16mono',-0.40,0.0062),('B_ASC_IgM',0.31,0.042),('T CD8 CM',-0.50,3.7e-4)]:
    check(f'anchor {f} rho {rho} q {q}', round(S.loc[f,'rho_Ext'],2)==rho and np.isclose(S.loc[f,'q_sev_Ext'], q, rtol=0.06), (S.loc[f,'rho_Ext'], S.loc[f,'q_sev_Ext']))

## 3. Results 본문 수치 (focus populations)

In [ ]:
ci = CI[CI.CLL_excluded].set_index(['feature','set'])
def r2(v): return round(float(v), 2)
check('IgG Real rho .36 P .029 / New .42 P .023', (r2(ci.loc[('B_ASC_IgG','Real'),'rho']), round(ci.loc[('B_ASC_IgG','Real'),'p_sev'],3), r2(ci.loc[('B_ASC_IgG','PBISC new'),'rho']), round(ci.loc[('B_ASC_IgG','PBISC new'),'p_sev'],3)) == (0.36,0.029,0.42,0.023))
check('IgA Real rho .38 P .021 / New .39 P .037', (r2(ci.loc[('B_ASC_IgA','Real'),'rho']), round(ci.loc[('B_ASC_IgA','Real'),'p_sev'],3), r2(ci.loc[('B_ASC_IgA','PBISC new'),'rho']), round(ci.loc[('B_ASC_IgA','PBISC new'),'p_sev'],3)) == (0.38,0.021,0.39,0.037))
check('IgG/IgA q Real .097/.074, Ext q .0071/.0078', np.allclose([S.loc['B_ASC_IgG','q_sev_Real'],S.loc['B_ASC_IgA','q_sev_Real'],S.loc['B_ASC_IgG','q_sev_Ext'],S.loc['B_ASC_IgA','q_sev_Ext']],[0.097,0.074,0.0071,0.0078],rtol=0.05))
sc = SC.set_index(['feature','outcome'])
check('P_het IgG .77 IgA .97, CM death .70, CTL sev .035, HSPC .97',
      [round(sc.loc[k,'p_het'],2) for k in [('B_ASC_IgG','sev'),('B_ASC_IgA','sev'),('T CD8 CM','death'),('HSPC','death')]]==[0.77,0.97,0.70,0.97] and round(sc.loc[('T CD8 CTL','sev'),'p_het'],3)==0.035)
check('robust 14/14 IgG, IgA, CTL sev; 12/14 CTL death; 13/14 HSPC',
      [int(sc.loc[k,'robust_pass']) for k in [('B_ASC_IgG','sev'),('B_ASC_IgA','sev'),('T CD8 CTL','sev'),('T CD8 CTL','death'),('HSPC','death')]]==[14,14,14,12,13])
check('CTL shared rho .55 P .0030', (r2(sc.loc[('T CD8 CTL','sev'),'shared_rho']), round(sc.loc[('T CD8 CTL','sev'),'shared_p'],4))==(0.55,0.003))
check('WHO2-3 CM .46 / CTL .30, P>=.069; HSPC .77 P .015',
      [r2(sc.loc[('T CD8 CM','death'),'AUC_within_WHO23']), r2(sc.loc[('T CD8 CTL','death'),'AUC_within_WHO23']), r2(sc.loc[('HSPC','death'),'AUC_within_WHO23']), round(sc.loc[('HSPC','death'),'p_within_WHO23'],3)]==[0.46,0.30,0.77,0.015]
      and min(sc.loc[('T CD8 CM','death'),'p_within_WHO23'], sc.loc[('T CD8 CTL','death'),'p_within_WHO23'])>=0.0685)
nf = SC[~SC.followed]
check('not-followed shared rho 0.07-0.19, P>=0.34', r2(nf.shared_rho.min())==0.07 and r2(nf.shared_rho.max())==0.19 and nf.shared_p.min()>=0.34)
print(SC[['feature','outcome','followed','shared_rho','shared_p']].round(3))

## 4. HSPC 사망 강건성 (Fig. 7l)

In [ ]:
h = HS.set_index(['analysis','set'])
E = 'Real + PBISC new'
exp = {'All deaths':(13,0.81,6.3e-4),'Table S1 deaths':(11,0.76,0.0066),'Direct COVID-19 deaths':(9,0.75,0.019),
       'Within WHO 2–3':(13,0.77,0.015),'Severity-stratified':(13,0.77,0.039),'Cambridge only (Real)':(12,0.85,4.6e-4),'Samples > day 7':(8,0.86,0.0011)}
for a,(d,auc,p) in exp.items():
    r = h.loc[(a,E)]
    check(f'{a}: {d} deaths AUC {auc} P {p}', int(r.deaths)==d and round(r.AUC,2)==auc and np.isclose(r.p,p,rtol=0.06), (r.deaths, r.AUC, r.p))
check('Real-only Table S1 P .13 (5 deaths), Direct P .36 (3 deaths)', (int(h.loc[('Table S1 deaths','Real'),'deaths']), round(h.loc[('Table S1 deaths','Real'),'p'],2), int(h.loc[('Direct COVID-19 deaths','Real'),'deaths']), round(h.loc[('Direct COVID-19 deaths','Real'),'p'],2))==(5,0.13,3,0.36))

### 4b. "uniquely among all 43 cell types — predicted death within WHO 2–3" 확인
같은 within-source normal score로 43개 세포형 모두 WHO 2–3 내 사망 AUC를 계산합니다.

In [ ]:
sub = NS[NS.sev>=2]; labs = [c for c in NS.columns if c not in ('sev','fatal','src','CD16mono|lin')]
rows = []
for f in labs:
    y = sub[f]; m = y.notna(); d1 = sub.fatal[m]==1
    u = stats.mannwhitneyu(y[m][d1], y[m][~d1]); rows.append((f, u.statistic/(d1.sum()*(~d1).sum()), u.pvalue))
W23 = pd.DataFrame(rows, columns=['feature','AUC_WHO23','p']).sort_values('p').reset_index(drop=True)
print(f'n={len(sub)}, deaths={int(sub.fatal.sum())}, cell types={len(labs)}'); print(W23.head(8).round(4))
sig = W23[W23.p<0.05].feature.tolist()
check('HSPC unique among 43 (WHO 2-3 P<0.05)', sig==['HSPC'], sig)
s4 = set(SEL.feature[SEL.S4_sev|SEL.S4_death])
check('HSPC unique among the 12 step-4 subsets', [f for f in sig if f in s4]==['HSPC'])

## 5. 모달리티 스케일 (Supp. Fig. 1i)와 raw % 병합

In [ ]:
M = MOD.set_index('feature')
txt = {'B_ASC_IgA':(9.4,1.5e-8),'B_ASC_dividing':(4.6,1.1e-4),'HSPC':(3.6,2.7e-4),'CD16mono':(2.8,0.0042),'T CD8 CTL':(1.4,0.0070),'NK CD56 bright':(0.34,0.0065)}
for f,(fold,p) in txt.items():
    fo = M.loc[f,'fold_PBISC_over_Real']; pw = M.loc[f,'wilcoxon_paired_p']
    check(f'{f}: fold {fold}x, Wilcoxon P {p:g} (text)', round(fo, 2 if fold<1 else 1)==fold and np.isclose(pw,p,rtol=0.05), f'CSV fold {fo:.2f}, P {pw:.2g}')
check('6 of 9 subsets with paired P<0.05', int((M.wilcoxon_paired_p<0.05).sum())==6)
check('IgA naive raw pooled rho +0.31; NK CD56bright raw -0.38, NS -0.50 P 1.6e-5',
      (r2(M.loc['B_ASC_IgA','combined_rho_raw_pct']), r2(M.loc['NK CD56 bright','combined_rho_raw_pct']), r2(M.loc['NK CD56 bright','combined_rho_normal_score']))==(0.31,-0.38,-0.50)
      and np.isclose(M.loc['NK CD56 bright','combined_p_sev_normal_score'],1.6e-5,rtol=0.05))
six = [M.loc['B_ASC_IgG','combined_p_sev_raw_pct'], M.loc['B_ASC_IgA','combined_p_sev_raw_pct'], M.loc['T CD8 CTL','combined_p_sev_raw_pct'],
       M.loc['T CD8 CM','combined_p_death_raw_pct'], M.loc['T CD8 CTL','combined_p_death_raw_pct'], M.loc['HSPC','combined_p_death_raw_pct']]
print(np.round(six,5)); check('six raw-% pooled P in 7.6e-4..0.024', np.isclose(min(six),7.6e-4,rtol=0.05) and round(max(six),3)==0.024)
rg = RAW.groupby(['src','sev']).B_ASC_IgA.agg(['min','max']).round(2); print(rg)
check('IgA text: "mild PBISC new 0.5–21.9%"', np.isclose(rg.loc[('New',0.0),'min'],0.5,atol=0.05) and np.isclose(rg.loc[('New',0.0),'max'],21.9,atol=0.05),
      f"mild New {rg.loc[('New',0.0),'min']}–{rg.loc[('New',0.0),'max']}%; all New {RAW[RAW.src=='New'].B_ASC_IgA.min():.2f}–{RAW[RAW.src=='New'].B_ASC_IgA.max():.2f}%")
check('IgA text: "critical Real 0.05–4.5%"', np.isclose(rg.loc[('Real',3.0),'min'],0.05,atol=0.01), f"critical Real {rg.loc[('Real',3.0),'min']}–{rg.loc[('Real',3.0),'max']}%")

## 6. CLL 제외 효과 · 검정력 · CD14 mono IFN

In [ ]:
ca = CI.set_index(['CLL_excluded','feature','set'])
check('dividing rho +0.39 -> +0.54; NK CD56bright -0.47 -> -0.55',
      [r2(ca.loc[(False,'B_ASC_dividing','Real'),'rho']), r2(ca.loc[(True,'B_ASC_dividing','Real'),'rho']), r2(ca.loc[(False,'NK CD56 bright','Real'),'rho']), r2(ca.loc[(True,'NK CD56 bright','Real'),'rho'])]==[0.39,0.54,-0.47,-0.55])
p20 = PW[PW.n_real==20].set_index(['feature','outcome'])
exp = {('B_ASC_IgG','sev'):(29,96,15),('B_ASC_IgA','sev'):(34,97,16),('T CD8 CM','death'):(9,71,None),('HSPC','death'):(41,99,18)}
for k,(a,b,c) in exp.items():
    r = p20.loc[k]; got = (round(100*r.real_alone), round(100*r.plus_pbisc_new), round(100*r.plus_shuffled))
    r100 = (100*r.real_alone, 100*r.plus_pbisc_new, 100*r.plus_shuffled)
    check(f'power n=20 {k}: {a}->{b}% (shuffled {c})', abs(r100[0]-a)<=0.51 and abs(r100[1]-b)<=0.51 and (c is None or abs(r100[2]-c)<=0.51), r100)
def n80(g, col):
    g = g.sort_values('n_real'); x, y = g.n_real.values, g[col].values
    if y[0] >= .8: return '<=10'
    for i in range(1,len(x)):
        if y[i] >= .8: return round(x[i-1]+(.8-y[i-1])*(x[i]-x[i-1])/(y[i]-y[i-1]))
    return 'n.r.'
n = {k:(n80(g,'real_alone'), n80(g,'plus_pbisc_new')) for k,g in PW.groupby(['feature','outcome'])}
check('n80 IgG n.r.->≤10, IgA 33->≤10, HSPC 31->≤10', [n[('B_ASC_IgG','sev')], n[('B_ASC_IgA','sev')], n[('HSPC','death')]]==[('n.r.','<=10'),(33,'<=10'),(31,'<=10')], n)
sa = SA[(SA.feature=='CD14mono_IFN')&(SA.set==E)].set_index('variant')
check('CD14 mono IFN AUC .75 P .0048; first sample .78 P .002; centre adj P .071',
      (r2(sa.loc['0 baseline (no CLL)','AUC']), round(sa.loc['0 baseline (no CLL)','p_death'],4), r2(sa.loc['F1 first sample per patient','AUC']), round(sa.loc['F1 first sample per patient','p_death'],3), round(sa.loc['A1 adjust centre/wave','p_death'],3))==(0.75,0.0048,0.78,0.002,0.071))

## 7. Supp. Fig. 2a 포함 세포형

In [ ]:
s = SCAN[SCAN.CLL_excluded]; hit = sorted(s[(s.q_sev<0.05)|(s.q_death<0.10)].feature.unique())
LBL = ['B_ASC_dividing','NK CD56 bright','HSPC','T CD8 CM','T CD8 CTL','B_ASC_IgG','B_ASC_IgA','B_ASC_IgM','CD14mono_IFN','CD14mono_dex','CD16mono','CD16mono_C1','B_CD11c','DC1','DC2','DC3','NK','NKT','T CD4 CTL','T g/d','MAIT']
print(len(hit), 'eligible;', len([f for f in hit if f in LBL]), 'plotted')
check('every FDR hit (any set) is plotted in S2a', set(hit) <= set(LBL), sorted(set(hit)-set(LBL)))
print(s[s.feature=='Int.mono'][['set','rho','q_sev','AUC_death','q_death']].round(4))

## 8. 원본 입력부터 재계산
**8a.** `run_ver8_pipeline.py`를 원본 입력(`eskd_covid_clinical_assoc_20261115/results`, `paper_claim_analysis`)으로 다시 실행했고, 출력 CSV 13개가 저장본과 최대 1.8×10⁻¹⁵ 차이로 일치했습니다 (난수 seed 동일 → bootstrap·검정력·permutation도 동일).

**8b.** 아래 셀은 파이프라인 코드를 쓰지 않고 다시 구현한 `verify_independent_recompute.py`를 실행합니다(환자 표 구성, 43개 스캔, BH, 5단계 퍼널과 centre 보정, 27명 Wilcoxon, HSPC 강건성, 다른 seed의 검정력). `PBISC_ROOT`는 `eskd_covid_clinical_assoc_20261115/`와 `HSPC_CD56brightNK_COVID_analysis_summary_20261116/`가 있는 폴더입니다.

In [ ]:
import runpy
_ = runpy.run_path(os.environ.get('V8_INDEP', 'verify_independent_recompute.py'), run_name='__main__')

## 요약

In [ ]:
print(f'{len(issues)} 불일치'); 
for l,d in issues: print('-', l, '|', d)